# Практика · Межі RNN

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.html](homework.html)

> ⏱ Зошит навчає 27 маленьких мереж і робить чотири заміри часу.
> Заміряно: **близько трьох хвилин процесорного часу** на чотирьох ядрах без
> відеокарти. Стінного часу на **вільній** машині виходить менше — близько двох хвилин, бо
> частину дослідів зошит навмисне рахує в чотири потоки; на завантаженій машині
> буває вчетверо більше. Зошит друкує і її навантаження, і свій власний
> процесорний час останньою клітинкою.

Тема 19 показала, що якість seq2seq падає з довжиною джерела. Тема 18 показала,
що градієнт до першого слова затухає. Тут ми зводимо межі рекурентних мереж
докупи й **міряємо** кожну — зокрема ту, яку підручники зазвичай наводять без числа:
«RNN не розпаралелюється, тому вона повільна».

Що зробимо:

1. подивимось на корпус: чому наші речення не показують проблеми довгих залежностей;
2. заміряємо, як градієнт доходить до першого кроку на 10, 25, 50 і 100 кроках;
3. заміряємо **вузьке горло** — скільки інформації влазить в один вектор сталого розміру;
4. напишемо крок LSTM руками й звіримо з бібліотечним — щоб побачити залежність від `t−1`;
5. **дослід А**: однакова робота, різна довжина, один потік;
6. **дослід Б**: та сама робота на одному й на чотирьох потоках;
7. **дослід В**: звідки береться різниця — та сама робота, поділена на k послідовних шматків;
8. **дослід Г**: де рекурентна мережа виграє — потокова обробка.

## 0 · Середовище

Грабля курсу №30: без фіксації потоків процесорний час бреше — потоки OpenMP
крутяться в очікуванні, і це очікування рахується як робота. Тому змінні
середовища ставимо **до** імпорту `numpy` і `torch`.

Стінний час теж друкуємо, але завжди поруч із навантаженням машини — інакше
число неперевірюване.

In [ ]:
import os
# фіксуємо потоки ДО імпорту numpy і torch — інакше процесорний час бреше
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import sys, time, math, glob, gettext, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(1)

print('Python      ', sys.version.split()[0])
print('numpy       ', np.__version__)
print('torch       ', torch.__version__)
print('ядер        ', os.cpu_count())
print('відеокарта  ', 'є' if torch.cuda.is_available() else 'немає')
print('навантаження машини (1 хв):', round(os.getloadavg()[0], 2))

## 1 · Корпус: чому саме на ньому проблема не видно

Наш корпус — українські переклади інтерфейсів із системи. Для цієї теми в ньому
важлива одна властивість: **довжина речення**. Якщо речення короткі, то ані
затухання градієнта, ані вузьке горло на них не проявляться — і це треба сказати
чесно, перш ніж міряти.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"   # канонічний токенізатор курсу

def load_corpus():
    '''Читає українські каталоги перекладів, що лежать у самій системі.'''
    texts = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue
        for source, target in catalog._catalog.items():
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                texts.append(target)
    return texts

corpus = load_corpus()
lengths = np.array([len(re.findall(TOKEN_PATTERN, t.lower())) for t in corpus])
lengths = lengths[lengths > 0]

print('документів у корпусі:', len(corpus))
print('медіана довжини:     ', int(np.median(lengths)), 'слів')
print('90-й перцентиль:     ', int(np.percentile(lengths, 90)), 'слів')
print('найдовше речення:    ', int(lengths.max()), 'слів')
for limit in (8, 16, 32, 100):
    print(f'  речень не довших за {limit:3d} слів: {(lengths <= limit).mean():.4f}')

**Що це означає.** Половина речень корпусу коротша за сім слів, і майже вісім із
десяти вміщаються у вісім. Рекурентна мережа на такому тексті майже ніколи не
доходить до тієї довжини, на якій її слабкі місця стають видні. Тому далі ми
міряємо межі **не на цьому корпусі**, а на штучних послідовностях потрібної
довжини: там, де перевіряється формула чи будова, синтетика чесніша за реальні
дані, бо дає рівно ту довжину, яку ми хочемо перевірити.

## 2 · Межа перша: градієнт до першого кроку

Беремо мережу **при ініціалізації**, подаємо послідовність довжини `T`, штовхаємо
похідну від останнього виходу назад і дивимось, яка норма градієнта доходить до
**входу першого кроку** порівняно з входом останнього.

Число, яке рахуємо, — це відношення «норма на кроці T» ділити на «норма на кроці 1».
Що воно більше, то слабший сигнал доходить до початку речення.

Це той самий замір, який докладно розбирає тема `18-lstm-gru`; тут ми повторюємо
його своїм зошитом, щоб число в цій лекції було власним. Конфігурація наша:
`H = 64`, три зерна, мережа не навчена.

In [ ]:
HIDDEN_GRAD = 64

def gradient_reach(kind, T, seed):
    '''Відношення норми градієнта на останньому кроці до норми на першому.'''
    torch.manual_seed(seed)
    layer = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[kind](
        HIDDEN_GRAD, HIDDEN_GRAD, batch_first=True)
    x = torch.randn(1, T, HIDDEN_GRAD, requires_grad=True)
    y, _ = layer(x)
    y[0, -1].pow(2).sum().backward()          # штовхаємо сигнал від останнього кроку
    per_step = x.grad[0].norm(dim=1)
    first, last = per_step[0].item(), per_step[-1].item()
    return float('inf') if first == 0.0 else last / first

print(f"{'T':>5} {'RNN':>14} {'LSTM':>14} {'GRU':>14}   (у скільки разів слабший)")
grad_table = {}
for T in (10, 25, 50, 100):
    row = []
    for kind in ('RNN', 'LSTM', 'GRU'):
        ratios = [gradient_reach(kind, T, seed) for seed in (0, 1, 2)]
        row.append(np.median(ratios))
    grad_table[T] = row
    print(f'{T:>5} ' + ' '.join(f'{v:>14.4g}' for v in row))

zeroed = [gradient_reach('RNN', 100, s) for s in (0, 1, 2)]
print('\nRNN на 100 кроках, усі три зерна:', ['нуль' if v == float('inf') else round(v, 2) for v in zeroed])

**Читай так.** `inf` означає, що градієнт до першого кроку виявився **точно нулем**
у float32 — не «малим», а нулем: числа скінчилися. Гейти LSTM і GRU затухання не
скасовують, а вповільнюють: на 50 кроках вони теж віддають мільярди разів слабший
сигнал, просто на кілька порядків менше, ніж проста RNN.

## 3 · Межа друга: одне вузьке горло

У seq2seq енкодер стискає **всю** вхідну фразу в один вектор сталого розміру, і
декодер бачить лише його. Тема `19-seq2seq` заміряла наслідок на справжніх
парах перекладу. Тут ми міряємо саму будову — на задачі, у якій немає нічого,
крім памʼяті.

**Задача-копія.** Мережа читає послідовність випадкових символів, стискає її в
один вектор, а потім має вимовити ту саму послідовність назад. Символи випадкові
й рівноймовірні, тобто **жодної надмірності**: угадати нічого не можна, можна
тільки запамʼятати. Це навмисно найважчий випадок — у справжньому тексті сусідні
слова підказують одне одного, тому там усе не так різко.

In [ ]:
VOCAB_COPY = 32        # скільки різних символів
EMB_COPY = 32
BATCH_COPY = 64
STEPS_COPY = 200       # кроків навчання на кожну конфігурацію

class Copier(nn.Module):
    '''Енкодер стискає всю послідовність в один вектор, декодер її відновлює.'''
    def __init__(self, hidden):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_COPY + 1, EMB_COPY)   # +1 — стартовий символ
        self.encoder = nn.GRU(EMB_COPY, hidden, batch_first=True)
        self.decoder = nn.GRU(EMB_COPY, hidden, batch_first=True)
        self.head = nn.Linear(hidden, VOCAB_COPY)

    def forward(self, seq):
        _, context = self.encoder(self.emb(seq))            # ось воно, вузьке горло
        start = torch.full((seq.shape[0], 1), VOCAB_COPY, dtype=torch.long)
        shifted = torch.cat([start, seq[:, :-1]], dim=1)    # подаємо еталон (teacher forcing)
        out, _ = self.decoder(self.emb(shifted), context)
        return self.head(out)

def copy_accuracy(length, hidden, seed):
    '''Навчає копіювальник і повертає частку правильно відтворених символів.'''
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed + 100)
    model = Copier(hidden)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(STEPS_COPY):
        seq = torch.randint(0, VOCAB_COPY, (BATCH_COPY, length), generator=gen)
        opt.zero_grad()
        loss_fn(model(seq).reshape(-1, VOCAB_COPY), seq.reshape(-1)).backward()
        opt.step()
    with torch.no_grad():
        seq = torch.randint(0, VOCAB_COPY, (512, length), generator=gen)
        return (model(seq).argmax(-1) == seq).float().mean().item()

started = time.process_time()
print('вектор контексту сталий: H = 64;  випадкове вгадування дало б', round(1 / VOCAB_COPY, 4))
copy_by_length = {}
for length in (4, 8, 16, 32):
    accs = [copy_accuracy(length, 64, s) for s in (0, 1, 2)]
    copy_by_length[length] = accs
    print(f'довжина {length:3d}:  частка правильних {np.median(accs):.4f} '
          f'[{min(accs):.4f} … {max(accs):.4f}]')
print('процесорний час розділу:', round(time.process_time() - started, 1), 'с')

Тепер навпаки: довжину тримаємо сталою (вісім символів — рівно та довжина
джерела, на якій тема `19-seq2seq` заміряла найгірший результат), а міняємо
**розмір самого вектора контексту**. Якщо справа у вузькому горлі, ширший вектор
має рятувати.

In [ ]:
started = time.process_time()
print('довжина стала: 8 символів;  міняємо розмір вектора контексту')
copy_by_hidden = {}
for hidden in (16, 32, 64, 128, 256):
    accs = [copy_accuracy(8, hidden, s) for s in (0, 1, 2)]
    copy_by_hidden[hidden] = accs
    print(f'H = {hidden:4d}:  частка правильних {np.median(accs):.4f} '
          f'[{min(accs):.4f} … {max(accs):.4f}]')
print('процесорний час розділу:', round(time.process_time() - started, 1), 'с')

## 4 · Крок LSTM руками: звідки береться «послідовність»

Перш ніж міряти час, подивимось на формулу. Один крок LSTM рахує чотири гейти з
**двох** доданків: одного від входу `x` і одного від попереднього стану `h`.
Другий доданок і є вся суть: щоб порахувати крок `t`, треба вже мати крок `t−1`.

Напишемо крок самі й звіримо з бібліотечним — щоб переконатися, що всередині
`nn.LSTMCell` немає нічого, крім цих пʼяти рядків.

In [ ]:
def lstm_step_by_hand(x, h_prev, c_prev, cell):
    '''Один крок LSTM: чотири гейти, з них два доданки — від входу і від h_prev.'''
    gates = x @ cell.weight_ih.T + cell.bias_ih + h_prev @ cell.weight_hh.T + cell.bias_hh
    size = cell.hidden_size
    i = torch.sigmoid(gates[:, 0*size:1*size])      # вхідний гейт
    f = torch.sigmoid(gates[:, 1*size:2*size])      # гейт забування
    g = torch.tanh(gates[:, 2*size:3*size])         # кандидат у памʼять
    o = torch.sigmoid(gates[:, 3*size:4*size])      # вихідний гейт
    c_next = f * c_prev + i * g
    return o * torch.tanh(c_next), c_next

torch.manual_seed(0)
cell = nn.LSTMCell(8, 8)
x0 = torch.randn(3, 8)
h0, c0 = torch.zeros(3, 8), torch.zeros(3, 8)
with torch.no_grad():
    ours_h, ours_c = lstm_step_by_hand(x0, h0, c0, cell)
    lib_h, lib_c = cell(x0, (h0, c0))
assert torch.allclose(ours_h, lib_h, atol=1e-6), 'стан h розійшовся!'
assert torch.allclose(ours_c, lib_c, atol=1e-6), 'памʼять c розійшлась!'
print('✅ наш крок LSTM збігається з nn.LSTMCell, максимальне відхилення',
      f'{(ours_h - lib_h).abs().max().item():.2e}')

# А тепер те саме питання до уваги: чи залежить позиція від попередньої?
def attention_by_hand(q, k, v):
    '''Одноголова увага: кожна позиція дивиться на всі одразу, без черги.'''
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
    return torch.softmax(scores, dim=-1) @ v

torch.manual_seed(0)
q, k, v = torch.randn(2, 6, 16), torch.randn(2, 6, 16), torch.randn(2, 6, 16)
assert torch.allclose(attention_by_hand(q, k, v),
                      F.scaled_dot_product_attention(q, k, v), atol=1e-6)
print('✅ наша увага збігається з F.scaled_dot_product_attention')

# Пряма перевірка того, що в увазі немає черги: переставимо позиції входу.
# Якщо позиція не залежить від попередньої, кожна дістане той самий вихід —
# лише переїде разом зі своїм місцем.
order = torch.randperm(6)
straight = attention_by_hand(q, k, v)
shuffled = attention_by_hand(q[:, order], k[:, order], v[:, order])
assert torch.allclose(straight[:, order], shuffled, atol=1e-6), 'позиції не переставні!'
print('✅ переставили позиції — кожна дістала той самий вихід')
print('   дій, що мусять чекати одна на одну: у LSTM стільки, скільки слів;')
print('   в увазі три (проєкції → ваги → змішування), і це не залежить від довжини')

Ось і вся теза про розпаралелення, і вона **не про час**, а про будову: у формулі
LSTM стоїть `h_prev`, у формулі уваги — ні. Тому кроки LSTM мусять іти по черзі, а
позиції уваги можуть рахуватись одночасно.

Далі ми перевіряємо, **у що саме** ця будова обходиться на нашій машині — чотири
ядра, без відеокарти.

## 5 · Дослід А: однакова робота, різна довжина, один потік

Тримаємо добуток «батч × довжина» сталим: 8192 позицій у будь-якому рядку
таблиці. Роботи однаково — міняється лише те, як вона розкладена. Міряємо один
крок навчання: прямий хід, втрата, зворотний хід. Один потік, три зерна,
процесорний годинник.

In [ ]:
HIDDEN_A, TOTAL_A = 128, 8192

class AttentionBlock(nn.Module):
    '''Одноголова увага з чотирма проєкціями — щоб ваг було стільки ж порядком.'''
    def __init__(self, hidden):
        super().__init__()
        self.to_q = nn.Linear(hidden, hidden)
        self.to_k = nn.Linear(hidden, hidden)
        self.to_v = nn.Linear(hidden, hidden)
        self.to_out = nn.Linear(hidden, hidden)
        self.scale = 1.0 / math.sqrt(hidden)

    def forward(self, x):
        q, k, v = self.to_q(x), self.to_k(x), self.to_v(x)
        weights = torch.softmax(torch.bmm(q, k.transpose(1, 2)) * self.scale, dim=-1)
        return self.to_out(torch.bmm(weights, v))

def forward_out(model, x):
    '''nn.LSTM повертає пару, наш блок — тензор; зводимо до одного вигляду.'''
    y = model(x)
    return y[0] if isinstance(y, tuple) else y

def train_step_seconds(make_model, length, batch, hidden, seed, reps=3):
    '''Процесорний час одного кроку навчання, усереднений по reps повторах.'''
    torch.manual_seed(seed)
    model = make_model(hidden)
    x = torch.randn(batch, length, hidden)
    forward_out(model, x).sum().backward()      # прогрів: перший виклик дорожчий
    model.zero_grad()
    start = time.process_time()
    for _ in range(reps):
        forward_out(model, x).pow(2).mean().backward()
        model.zero_grad()
    return (time.process_time() - start) / reps

make_lstm = lambda hidden: nn.LSTM(hidden, hidden, batch_first=True)
started = time.process_time()

print('потоків:', torch.get_num_threads(), '· добуток батч × довжина = ', TOTAL_A)
print(f"{'довжина':>8}{'батч':>7}{'LSTM, с':>12}{'увага, с':>12}")
expA = {}
for length in (8, 32, 128, 256, 512, 1024):
    batch = TOTAL_A // length
    lstm_t = [train_step_seconds(make_lstm, length, batch, HIDDEN_A, s) for s in (0, 1, 2)]
    attn_t = [train_step_seconds(AttentionBlock, length, batch, HIDDEN_A, s) for s in (0, 1, 2)]
    # округлюємо ОДРАЗУ: далі всі відношення рахуються з тих самих чисел, які надруковані
    expA[length] = (round(float(np.median(lstm_t)), 4), round(float(np.median(attn_t)), 4))
    print(f'{length:>8}{batch:>7}{expA[length][0]:>12.4f}{expA[length][1]:>12.4f}')

base_l, base_a = expA[8]
print()
for length in (8, 32, 128, 256, 512, 1024):
    print(f'від довжини 8 до {length:4d}:  LSTM у {expA[length][0]/base_l:.2f} раза, '
          f'увага у {expA[length][1]/base_a:.2f} раза')
print('процесорний час розділу:', round(time.process_time() - started, 1), 'с')

**Це протилежне тому, що пишуть у підручниках.** На одному ядрі час LSTM від
довжини майже не залежить: роботи стільки ж, а те, що вона порізана на більше
кроків, коштує копійки. А от увага дорожчає — бо її вартість квадратична за
довжиною, і при сталій кількості позицій довші рядки означають більші матриці ваг.

Отже теза «RNN повільна, бо не розпаралелюється» тут не відтворюється. Шукаємо,
за яких умов вона все-таки має сенс.

## 6 · Дослід Б: та сама робота на одному й на чотирьох потоках

Беремо одну точку — `H = 512`, батч 32, довжина 128 — і рахуємо її двічі:
дозволивши один потік і дозволивши чотири. Друкуємо **обидва** годинники й
навантаження машини поруч, як вимагає грабля №30.

In [ ]:
HIDDEN_B, BATCH_B, LENGTH_B = 512, 32, 128

def train_step_both_clocks(make_model, threads, seed, reps=1):
    '''Повертає (стінний, процесорний) час одного кроку навчання.'''
    torch.set_num_threads(threads)
    torch.manual_seed(seed)
    model = make_model(HIDDEN_B)
    warm = torch.randn(4, LENGTH_B, HIDDEN_B)        # дешевий прогрів
    forward_out(model, warm).sum().backward()
    model.zero_grad()
    x = torch.randn(BATCH_B, LENGTH_B, HIDDEN_B)
    wall0, cpu0 = time.perf_counter(), time.process_time()
    for _ in range(reps):
        forward_out(model, x).pow(2).mean().backward()
        model.zero_grad()
    return (time.perf_counter() - wall0) / reps, (time.process_time() - cpu0) / reps

started = time.process_time()
print('навантаження машини на початок досліду:', round(os.getloadavg()[0], 2))
expB = {}
for name, maker in (('LSTM', make_lstm), ('увага', AttentionBlock)):
    for threads in (1, 4):
        pairs = [train_step_both_clocks(maker, threads, s) for s in (0, 1, 2)]
        wall = round(float(np.median([p[0] for p in pairs])), 4)
        cpu = round(float(np.median([p[1] for p in pairs])), 4)
        expB[(name, threads)] = (wall, cpu)
        print(f'{name:>6}, потоків {threads}:  стінний {wall:.4f} с   процесорний {cpu:.4f} с')
torch.set_num_threads(1)

print('\nщо дали чотири ядра замість одного:')
penalty = {}
for name in ('LSTM', 'увага'):
    w1, c1 = expB[(name, 1)]
    w4, c4 = expB[(name, 4)]
    penalty[name] = round(c4 / c1, 2)
    print(f'  {name:>6}: стінний у {w4/w1:.2f} раза, процесорний у {c4/c1:.2f} раза')
print(f'  покарання LSTM більше за покарання уваги у '
      f'{penalty["LSTM"]/penalty["увага"]:.2f} раза')
print('навантаження машини на кінець досліду:', round(os.getloadavg()[0], 2))
print('процесорний час розділу:', round(time.process_time() - started, 1), 'с')

## 7 · Дослід В: звідки береться різниця

Здогад простий: LSTM видає **128 дрібних послідовних операцій** і платить
синхронізацію потоків на кожній, а увага — кілька великих і платить кілька разів.

Перевіримо це без жодних мереж. Візьмемо один множник матриць, роботу в якому
можна порізати на `k` послідовних шматків. Роботи щоразу **рівно стільки ж** —
міняється лише кількість викликів.

In [ ]:
ROWS_E, WIDTH_E, REPS_E = 4096, 512, 2

def chunked_matmul_seconds(chunks, threads, seed):
    '''Процесорний час одного й того самого множення, порізаного на chunks частин.'''
    torch.set_num_threads(threads)
    torch.manual_seed(seed)
    x = torch.randn(ROWS_E, WIDTH_E)
    w = torch.randn(WIDTH_E, WIDTH_E)
    rows = ROWS_E // chunks
    for i in range(chunks):                       # прогрів
        x[i*rows:(i+1)*rows] @ w
    start = time.process_time()
    for _ in range(REPS_E):
        for i in range(chunks):
            x[i*rows:(i+1)*rows] @ w
    return (time.process_time() - start) / REPS_E

started = time.process_time()
print(f'однакова робота {ROWS_E}×{WIDTH_E}×{WIDTH_E}, порізана на k послідовних шматків')
print(f"{'k':>5}{'1 потік, мс':>14}{'4 потоки, мс':>15}{'програш':>10}")
expE = {}
for chunks in (1, 2, 8, 32, 128):
    one = round(float(np.median([chunked_matmul_seconds(chunks, 1, s) for s in (0, 1, 2)])) * 1000, 2)
    four = round(float(np.median([chunked_matmul_seconds(chunks, 4, s) for s in (0, 1, 2)])) * 1000, 2)
    expE[chunks] = (one, four)          # мілісекунди, вже округлені
    print(f'{chunks:>5}{one:>14.2f}{four:>15.2f}{four/one:>9.2f}x')
torch.set_num_threads(1)
print(f'\nвід k = 1 до k = 128:  один потік у {expE[128][0]/expE[1][0]:.2f} раза, '
      f'чотири потоки у {expE[128][1]/expE[1][1]:.2f} раза')
print('процесорний час розділу:', round(time.process_time() - started, 1), 'с')

**Ось воно.** Одному потоку майже байдуже, на скільки шматків порізана робота.
Чотирьом — не байдуже зовсім: кожен шматок коштує окремої роздачі завдань і
окремого збору результатів, і на дрібних шматках ця плата перевищує саму роботу.

`k = 128` — це рівно довжина послідовності з досліду Б. Тобто LSTM там платив
за синхронізацію 128 разів, а увага — кілька.

## 8 · Дослід Г: де рекурентна мережа виграє

Тепер чесно в інший бік. Уявімо, що текст приходить **по одному слову** й треба
відповідати одразу — розпізнавання мовлення, підказка при наборі, датчик на
пристрої без відеокарти. Рекурентна мережа несе стан сталого розміру, тож кожен
новий крок коштує однаково. Увага мусить тримати всі попередні позиції й
переглядати їх щоразу.

In [ ]:
HIDDEN_S, STREAM_STEPS = 256, 2048

def stream_rnn(seed):
    '''Час кожного кроку, коли стан несеться далі без перегляду минулого.'''
    torch.manual_seed(seed)
    cell = nn.LSTMCell(HIDDEN_S, HIDDEN_S)
    h, c = torch.zeros(1, HIDDEN_S), torch.zeros(1, HIDDEN_S)
    x = torch.randn(1, HIDDEN_S)
    per_step = []
    with torch.no_grad():
        cell(x, (h, c))
        for _ in range(STREAM_STEPS):
            t0 = time.process_time()
            h, c = cell(x, (h, c))
            per_step.append(time.process_time() - t0)
    return np.array(per_step)

class StreamingAttention(nn.Module):
    '''Увага з кешем: кожен новий крок дописує ключ і значення й переглядає всі.'''
    def __init__(self, hidden):
        super().__init__()
        self.to_q = nn.Linear(hidden, hidden)
        self.to_k = nn.Linear(hidden, hidden)
        self.to_v = nn.Linear(hidden, hidden)
        self.to_out = nn.Linear(hidden, hidden)
        self.scale = 1.0 / math.sqrt(hidden)

    def step(self, x, keys, values):
        keys = torch.cat([keys, self.to_k(x)], dim=0)
        values = torch.cat([values, self.to_v(x)], dim=0)
        weights = torch.softmax((self.to_q(x) @ keys.T) * self.scale, dim=-1)
        return self.to_out(weights @ values), keys, values

def stream_attention(seed):
    torch.manual_seed(seed)
    model = StreamingAttention(HIDDEN_S)
    x = torch.randn(1, HIDDEN_S)
    keys, values = torch.zeros(0, HIDDEN_S), torch.zeros(0, HIDDEN_S)
    per_step = []
    with torch.no_grad():
        model.step(x, keys, values)
        for _ in range(STREAM_STEPS):
            t0 = time.process_time()
            _, keys, values = model.step(x, keys, values)
            per_step.append(time.process_time() - t0)
    return np.array(per_step)

BUCKETS = 8                     # ділимо 2048 кроків на вісім рівних відрізків
started = time.process_time()
stream, stream_curve = {}, {}
for name, fn in (('RNN', stream_rnn), ('увага з кешем', stream_attention)):
    runs = np.vstack([fn(seed) * 1e6 for seed in (0, 1, 2)])        # 3 × 2048, мікросекунди
    by_bucket = runs.reshape(3, BUCKETS, STREAM_STEPS // BUCKETS)
    curve = np.round(np.median(np.median(by_bucket, axis=2), axis=0), 1)   # медіана всередині, медіана по зернах
    totals = runs.sum(axis=1) / 1e6
    stream_curve[name] = curve
    stream[name] = (np.median(totals), curve[0], curve[-1])
    print(f'{name:>14}: усього {np.median(totals):.4f} с   '
          f'перший відрізок {curve[0]:.1f} мкс   останній {curve[-1]:.1f} мкс   '
          f'ріст {curve[-1]/curve[0]:.2f}x')
    print('                час кроку по відрізках, мкс: ' +
          '  '.join(f'{v:.1f}' for v in curve))

print(f'\nщо треба тримати в памʼяті на кроці {STREAM_STEPS}:')
print(f'  RNN:   стан 2 × H = {2*HIDDEN_S} чисел, і це не залежить від кроку')
print(f'  увага: кеш 2 × t × H = {2*STREAM_STEPS*HIDDEN_S} чисел, '
      f'тобто у {STREAM_STEPS} разів більше')
print('процесорний час розділу:', round(time.process_time() - started, 1), 'с')

## 9 · Підсумок числами

In [ ]:
print('МЕЖА 1 · градієнт до першого кроку, H = 64, при ініціалізації')
print(f'   на 100 кроках RNN:  {"нуль у float32" if grad_table[100][0] == float("inf") else round(grad_table[100][0])}')
print(f'   на 100 кроках LSTM: слабший у {grad_table[100][1]:.3g} раза')
print()
print('МЕЖА 2 · один вектор сталого розміру, задача-копія, H = 64')
print(f'   4 символи:  {np.median(copy_by_length[4]):.4f}')
print(f'   8 символів: {np.median(copy_by_length[8]):.4f}')
print(f'  16 символів: {np.median(copy_by_length[16]):.4f}  (випадкове вгадування {1/VOCAB_COPY:.4f})')
print(f'   а на восьми символах ширший вектор рятує: H = 16 дає '
      f'{np.median(copy_by_hidden[16]):.4f}, H = 256 дає {np.median(copy_by_hidden[256]):.4f}')
print()
print('МЕЖА 3 · будова залежностей, чотири ядра, без відеокарти')
print(f'   один потік, довжина 8 → 1024:  LSTM у {expA[1024][0]/expA[8][0]:.2f} раза, '
      f'увага у {expA[1024][1]/expA[8][1]:.2f} раза')
print(f'   чотири потоки замість одного (процесорний час): '
      f'LSTM у {expB[("LSTM",4)][1]/expB[("LSTM",1)][1]:.2f} раза гірше, '
      f'увага у {expB[("увага",4)][1]/expB[("увага",1)][1]:.2f} раза')
print(f'   та сама робота, порізана на 128 шматків: чотири потоки програють у '
      f'{expE[128][1]/expE[128][0]:.2f} раза, один — у {expE[128][0]/expE[1][0]:.2f}')
print()
print('ДЕ RNN ВИГРАЄ · потокова обробка, 2048 кроків по одному')
print(f'   час кроку RNN змінився у {stream["RNN"][2]/stream["RNN"][1]:.2f} раза, '
      f'уваги — у {stream["увага з кешем"][2]/stream["увага з кешем"][1]:.2f} раза')
print(f'   памʼять: {2*HIDDEN_S} чисел проти {2*STREAM_STEPS*HIDDEN_S}')
print()
print('процесорний час усього зошита:', round(time.process_time(), 1), 'с')

## Завдання

### 🟢 Рівень 1
Повтори дослід В із іншим розміром матриці — наприклад `ROWS_E = 8192`,
`WIDTH_E = 256`. Чи лишається програш на `k = 128` того самого порядку?
**Зроблено, якщо:** маєш таблицю з пʼятьма значеннями `k` і можеш назвати, при
якому `k` чотири потоки перестають програвати.

### 🟡 Рівень 2
У досліді А ми тримали `H = 128`. Повтори його при `H = 512` і подивись, чи
зсунеться довжина, на якій увага дорожчає за LSTM. Підказка: вартість уваги
складається з двох доданків, і лише один із них квадратичний за довжиною.
**Зроблено, якщо:** знайшов довжину перетину для обох `H` і пояснив словами,
чому вона зсунулась саме в цей бік.

### 🔴 Рівень 3
Задача-копія міряла найважчий випадок — символи без надмірності. Зроби
надмірність керованою: нехай кожен символ повторюється двічі підряд
(`aabbcc…`), тобто інформації вдвічі менше при тій самій довжині. Заміряй
частку правильних для довжин 8, 16, 32 при `H = 64`, три зерна.
**Зроблено, якщо:** можеш сказати числом, наскільки надмірність зсуває стелю,
і чи стеля визначається довжиною послідовності чи кількістю інформації в ній.